In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
transform=transforms.Compose([
    transforms.ToTensor(),
])
train_data=torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)
test_data=torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)


100%|██████████| 26.4M/26.4M [00:00<00:00, 116MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.29MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 54.2MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.1MB/s]


In [3]:
image,label=train_data[0]

print(image.shape)
print(label)

torch.Size([1, 28, 28])
9


In [4]:
import torch.nn as nn
import torch.nn.functional as F
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN,self).__init__()
        self.conv1=nn.Conv2d(1,32,kernel_size=3,padding=1)
        self.pool=nn.MaxPool2d(2,2)
        self.conv2=nn.Conv2d(32,64,kernel_size=3,padding=1)
        self.fc1=nn.Linear(64*7*7,128)
        self.fc2=nn.Linear(128,10)
    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=x.view(-1,64*7*7)
        x=F.relu(self.fc1(x))
        return self.fc2(x)
model=FashionCNN()
print(model)

FashionCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [5]:
from torch.utils.data import DataLoader

# Create the conveyor belt
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

In [6]:
import torch.optim as optim
# Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # Try lowering lr to 0.0001 if it still rises

epochs = 5
for epoch in range(epochs):
    running_loss = 0.0  # <--- RESET THIS EVERY EPOCH
    for images, labels in train_loader:
        # 1. Zero gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        outputs = model(images)
        
        # 3. Calculate loss
        loss = criterion(outputs, labels)
        
        # 4. Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    # Print the AVERAGE loss for the whole epoch
    avg_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}], Average Loss: {avg_loss:.4f}')

Epoch [1/5], Average Loss: 0.4575
Epoch [2/5], Average Loss: 0.2902
Epoch [3/5], Average Loss: 0.2419
Epoch [4/5], Average Loss: 0.2131
Epoch [5/5], Average Loss: 0.1899


In [7]:
# Set model to evaluation mode (turns off dropout/batchnorm updates)
model.eval() 

correct = 0
total = 0
test_loader = DataLoader(test_data, batch_size=64, shuffle=True)
# We use torch.no_grad() to save memory since we don't need gradients for testing
with torch.no_grad():
    for images, labels in test_loader: # Make sure you created a test_loader like train_loader
        outputs = model(images)
        
        # Get the class with the highest probability
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10,000 test images: {100 * correct / total:.2f}%')

Accuracy of the network on the 10,000 test images: 91.60%
